# Common similarity and dissimilarity measures: numeric example


### The measures, as a numbered map

1. **Similarity measures** for numeric features --- all three return values in $[-1, 1]$:
   - **Pearson** correlation: linear association between the raw values.
   - **Spearman** correlation: Pearson applied to the **ranks** of the values.
   - **Kendall's $\tau$**: agreement in the **ordering of pairs** of observations.
2. **Dissimilarity measures** --- distances, larger = farther:
   - **Euclidean** distance ($\ell_2$): square root of the summed squared differences.
   - **Manhattan** distance ($\ell_1$): summed absolute differences.
3. **The choice changes who counts as "close".** Similarity measures compare the *shape* of two profiles; distances compare their *values*. The same pair of points can be nearest neighbors under one measure and strangers under another --- the checks at the end of this notebook construct exactly that case.

> *Distance is an opinion: Euclidean asks how far apart two profiles sit, correlation asks whether they rise and fall together --- decide which question you are asking before you trust the answer.* --- Words of wisdom by Claude Fable 5 (2026)

In [ ]:
# --- Imports: the five measures from the numbered map above ----------------
# Map item 1 (similarity):    pearsonr, spearmanr, kendalltau  (scipy.stats)
# Map item 2 (dissimilarity): euclidean (l2), cityblock (= Manhattan, l1)
import numpy as np
from scipy.stats import pearsonr, spearmanr, kendalltau
from scipy.spatial.distance import euclidean, cityblock
import matplotlib.pyplot as plt

np.random.seed(42)   # fixed seed: everyone gets the same noise draws

def plot_all_scenarios(scenarios, titles):
    fig, axs = plt.subplots(1, 4, figsize=(24, 6))
    for i, (ax, (x, y), title) in enumerate(zip(axs, scenarios, titles)):
        # Map item 1: the three similarity measures for the pair (x, y)
        pearson = pearsonr(x, y)[0]      # linear association of raw values
        spearman = spearmanr(x, y)[0]    # Pearson on the ranks
        kendall = kendalltau(x, y)[0]    # agreement among orderings of pairs
        # Map item 2: the two distances for the same pair
        euclid = euclidean(x, y)         # l2: sqrt of summed squared diffs
        manhat = cityblock(x, y)         # l1: summed absolute diffs
        
        # Euclidean and Manhattan distances are also available in scikit-learn:
        # from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances

        # Plot data and y=x line
        ax.scatter(x, y)
        ax.plot([1, 10], [1, 10], 'r--', label='y = x')
        ax.set_title(title)
        ax.set_xlim(0, 11)
        ax.set_ylim(min(min(y), 0), max(max(y), 11))
        ax.set_xlabel("x")
        if i == 0:
            ax.set_ylabel("y")
        ax.grid(True)

        # Caption with increased font size
        # Map item 3: all five numbers side by side for the SAME pair --
        # compare them across scenarios to see the measures disagree.
        caption = (
            f"Pearson: {pearson:.2f}\n"
            f"Spearman: {spearman:.2f}\n"
            f"Kendall: {kendall:.2f}\n"
            f"Euclid: {euclid:.2f}\n"
            f"Manhattan: {manhat:.2f}"
        )
        ax.text(0.5, -0.25, caption, transform=ax.transAxes,
                fontsize=20, va='top', ha='center')

    plt.tight_layout()
    plt.show()


Recall that we have the following similarity measures:

-  Pearson correlation: This is the "usual" correlation coefficient, and measures the **linear** association between $X_i$ and $X_j$. 
-  Spearman correlation. This is essentially the Pearson correlation applied to **ranked** observations. 
-  Kendall's $\tau$. This correlation coefficient uses directly rankings among pairs of observations. 

and the dissimilarity measures

- Euclidean distance:  sum of squared difference between feature values ($\ell_2$-norm)
    $$ d_E(i,j)= \left[ \sum_{l=1}^p (X_{i,l}-X_{j,l})^2  \right]^{1/2} $$
- Manhattan distance:
	sum of absolute difference between feature values ($\ell_1$-norm)
    $$d_M(i,j)= \sum_{l=1}^p |X_{i,l}-X_{j,l}| $$

In [ ]:
# --- Data Generation ---
# Four scenarios, designed so the measures (map items 1 and 2) disagree:
x1 = np.arange(1, 11)
y1 = x1 + np.random.normal(0, 1.5, size=10)   # 1: linear + noise (Pearson's home turf)

x2 = np.arange(1, 11)
y2 = 10*np.sin(np.pi*(x2-1)/15)               # 2: monotone but curved (rank measures stay high)

x3 = np.ones(10)
y3 = np.random.normal(2, 1, size=10)
x3[-1] = 10; y3[-1]=7                          # 3: no relation + one extreme point (Pearson is dragged up)

x4 = np.arange(1, 11)
y4 = x4.copy().astype(float)
y4[4] = 9                                      # 4: perfect y = x except one blip

scenarios = [(x1, y1), (x2, y2), (x3, y3), (x4, y4)]
titles = [
    "1 ",
    "2 ",
    "3 ",
    "4 "
]

# --- One-liner to call it ---
plot_all_scenarios(scenarios, titles)
